# Phase 1: Dataset Exploration and Engineering

**Project:** Resume Screening and Skill Gap Analysis

This notebook explores the available datasets for the project to understand their structure, quality, and suitability for the core tasks: Resume parsing, Skill extraction, and Job matching.

In [ ]:
import os
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from datasets import load_dataset

# Add project root to sys.path
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

# Import custom utilities (assuming they exist)
try:
    from src.preprocessing.dataset_utils import *
    print("Successfully imported dataset_utils")
except ImportError:
    print("Could not import dataset_utils, will define necessary functions inline if needed.")

## 1. Dataset A1: EXAI-ResumeIntel Resumes

- **HF ID:** mithinsagar/exai-resumeintel-data
- **Config:** resumes
- **Rows:** 2,484 | **Columns:** 3 (ID, Category, Feature)

In [ ]:
print("Loading Dataset A1 (Resumes)...")
ds_a1 = load_dataset('mithinsagar/exai-resumeintel-data', 'resumes', split='train')
df_a1 = ds_a1.to_pandas()

print(f"Shape: {df_a1.shape}")
display(df_a1.head())

print("\nMissing values:")
print(df_a1.isnull().sum())

print(f"\nUnique IDs: {df_a1['ID'].nunique()}")

In [ ]:
# Category Distribution
plt.figure(figsize=(12, 6))
sns.countplot(y='Category', data=df_a1, order=df_a1['Category'].value_counts().index)
plt.title('Category Distribution in Dataset A1')
plt.xlabel('Count')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

category_counts = df_a1['Category'].value_counts()
print(f"Number of unique categories: {len(category_counts)}")
print(f"Imbalance ratio (max/min): {category_counts.max() / category_counts.min():.2f}")

## 2. Dataset A2: EXAI-ResumeIntel Job Postings (Streaming)

- **HF ID:** mithinsagar/exai-resumeintel-data
- **Config:** job_postings
- **Rows:** ~1.62 million (sampled for analysis)
- **Note:** All statistics below are SAMPLE-BASED (2,000 records).

In [ ]:
print("Loading Dataset A2 (Job Postings) in streaming mode...")
ds_a2_stream = load_dataset('mithinsagar/exai-resumeintel-data', 'job_postings', split='train', streaming=True)

# Sample 2000 records
sample_records = []
for i, record in enumerate(ds_a2_stream):
    if i >= 2000: break
    sample_records.append(record)

df_a2_sample = pd.DataFrame(sample_records)
print(f"Sample Shape: {df_a2_sample.shape}")
display(df_a2_sample.head())

print("\nMissing values in SAMPLE:")
print(df_a2_sample.isnull().sum())

In [ ]:
# Top 20 Roles in Sample
plt.figure(figsize=(10, 6))
sns.countplot(y='Role', data=df_a2_sample, order=df_a2_sample['Role'].value_counts().iloc[:20].index)
plt.title('Top 20 Roles Distribution in Dataset A2 (SAMPLE-BASED)')
plt.xlabel('Count')
plt.ylabel('Role')
plt.tight_layout()
plt.show()

print(f"Unique roles in sample: {df_a2_sample['Role'].nunique()}")

## 3. Dataset B: Resume-Job Fit Merged v1

- **HF ID:** med2425/resume-job-fit-merged-v1
- **Rows:** 80,017 (train), 13,716 (test)
- Contains pairings of resumes and JDs with a matching label.

In [ ]:
print("Loading Dataset B (Job Fit)...")
ds_b = load_dataset('med2425/resume-job-fit-merged-v1', split='train')
df_b = ds_b.to_pandas()

print(f"Shape: {df_b.shape}")
display(df_b.head())

print("\nMissing values:")
print(df_b.isnull().sum())

In [ ]:
# Label Distribution
plt.figure(figsize=(8, 5))
sns.countplot(x='label', data=df_b, order=df_b['label'].value_counts().index)
plt.title('Fit Label Distribution in Dataset B')
plt.show()

label_counts = df_b['label'].value_counts()
print("Label Counts:\n", label_counts)
print(f"Class imbalance ratio (No Fit vs Good Fit): {label_counts['No Fit'] / label_counts['Good Fit']:.2f}")

## 4. Dataset C: Resume NER Training Data

- **HF ID:** yashpwr/resume-ner-training-data
- **Rows:** 22,855
- **Format:** Chat-format (system/user/assistant messages), NOT traditional token-level NER.

In [ ]:
print("Loading Dataset C (NER)...")
ds_c = load_dataset('yashpwr/resume-ner-training-data', split='train')

print(f"Number of records: {len(ds_c)}")
print("\nSample record (first message):\n")
import pprint
pprint.pprint(ds_c[0]['messages'][:2])

## 5. Dataset Comparison Table

In [ ]:
comparison_data = {
    'Dataset': ['Dataset A1 (Resumes)', 'Dataset A2 (Job Postings)', 'Dataset B (Fit)', 'Dataset C (NER)'],
    'Size': [2484, '~1.62M', '80,017 (train)', 22855],
    'Primary Task': ['Classification/Extraction', 'Data Source', 'Job Matching', 'Information Extraction (LLM)'],
    'Format': ['Tabular (ID, Category, Text)', 'Tabular (Role, Features)', 'Tabular (Resume, JD, Label)', 'Chat Messages (List of Dicts)']
}
df_comparison = pd.DataFrame(comparison_data)
display(df_comparison)

## 6. Dataset Suitability Assessment

- **Dataset A1 (EXAI-ResumeIntel Resumes):** Highly suitable for building baseline classification models to categorize resumes into domains. Features are already somewhat processed (keywords), which may limit some deep NLP tasks that require full context.
- **Dataset A2 (EXAI-ResumeIntel Job Postings):** Massive source of job descriptions. Suitable for unsupervised pre-training, skill mining, or building a retrieval database.
- **Dataset B (Resume-Job Fit):** Essential for the core "matching" task. Provides supervised pairings. However, the `generated_smart` source implies synthetic pairs, which might not fully reflect real-world nuances. The severe class imbalance (10.4:1 No Fit vs Good Fit) needs addressing.
- **Dataset C (Resume NER):** Great for fine-tuning LLMs (like Llama/Mistral) for structured extraction since it uses a chat format rather than traditional token-level BIO tags.

## 7. Data Quality Summary

- **Missing Values:** Minimal across datasets (e.g., 1 missing feature in A1, 0 in B). 
- **Duplicates:** A1 has unique IDs. Need to monitor A2 and B for text-based duplicates during preprocessing.
- **Imbalance:** High imbalance in Dataset B labels and moderate imbalance across categories in A1.

## 8. Data Leakage Analysis

Since these are separate datasets, direct data leakage (train text appearing in test text) is a concern mainly if we mix them or if Dataset B's synthetic data generators used texts from Dataset A1. Before training unified models, cross-dataset deduplication or embedding similarity checks on test sets should be performed.

## 9. Train/Validation/Test Strategy

- **Dataset A1:** Standard 80/10/10 split stratified by `Category` to handle the mild imbalance.
- **Dataset B:** It already has a test split. Train should be split further into train/validation (e.g., 90/10) stratified by `label` to ensure `Good Fit` cases are represented.
- **Dataset C:** Hold out a small validation set of complete conversations to monitor extraction loss.

## 10. Large Dataset Strategy (Dataset A2)

For the 1.62M job postings:
1.  **Streaming:** Use `datasets` streaming mode to avoid loading it all into memory.
2.  **Sampling:** For initial feature engineering, take a stratified sample (e.g., 50k rows) based on `Role`.
3.  **Batch Processing:** For vector DB ingestion or massive skill extraction, use map functions with batched processing.

## 11. Conclusions and Recommendations for Phase 2

1.  **Imbalance Handling:** Implement SMOTE, class weighting, or downsampling for Dataset B prior to model training.
2.  **Preprocessing Pipeline:** Develop robust parsers for Dataset A1's keyword blobs vs Dataset B's full text.
3.  **Model Selection:** Given Dataset C's format, consider instruction-tuning an LLM for NER rather than training a traditional BiLSTM-CRF/BERT-token classifier.